### 1. conda 환경 생성 및 활성화
터미널에서 실행:  
conda create -n autogluon_env python=3.10 -y

conda activate autogluon_env

이후 환경 안에서 AutoGluon 설치:

pip install -U pip wheel setuptools

pip install autogluon.tabular -q 


In [3]:
# 파일: autogluon_run.py (예시)

import pandas as pd
from autogluon.tabular import TabularPredictor

# 1) 데이터 로드
train_path = "../data/train.tsv"
test_path = "../data/test.tsv"

# TSV 이므로 sep="\t" 사용
train_df = pd.read_csv(train_path, sep="\t")
test_df = pd.read_csv(test_path, sep="\t")


In [6]:

train_df = train_df.drop(columns=['train_id'])
test_df = test_df.drop(columns=['test_id'])


In [7]:

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# 2) 타깃 / ID 컬럼 이름 설정
TARGET_COL = "price"   # 실제 타깃 컬럼명으로 바꿔 주세요
ID_COL = None           # 제출용 ID 컬럼명으로 바꿔 주세요 (없으면 None)

# 3) AutoGluon 학습
# presets / time_limit 등은 상황에 맞게 조절
predictor = TabularPredictor(
    label=TARGET_COL,
    problem_type=None,          # 회귀/분류 자동 추론, 명시하고 싶으면 "regression"/"multiclass"/"binary"
    path="autogluon_models"     # 모델이 저장될 폴더
).fit(
    train_data=train_df,
    presets="medium_quality_faster_train",  # 빠른 실험용
    time_limit=3600,                        # 최대 1시간 (초 단위), 필요에 따라 조정
)

# 4) 리더보드 확인 (optional)
leaderboard_df = predictor.leaderboard(silent=True)
print(leaderboard_df.head())

# 5) 테스트 데이터 예측
test_preds = predictor.predict(test_df)

# 6) 제출 파일 생성
if ID_COL in test_df.columns:
    submission = pd.DataFrame({
        ID_COL: test_df[ID_COL],
        TARGET_COL: test_preds
    })
else:
    # ID 컬럼이 없다면 단순히 index 기반으로 생성
    submission = pd.DataFrame({
        "id": range(len(test_preds)),
        TARGET_COL: test_preds
    })

submission_path = "submission_autogluon.csv"
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path)

Preset alias specified: 'medium_quality_faster_train' maps to 'medium_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.10.19
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.19045
CPU Count:          12
Memory Avail:       24.24 GB / 31.91 GB (76.0%)
Disk Space Avail:   342.69 GB / 465.09 GB (73.7%)
Presets specified: ['medium_quality_faster_train']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 3600s
AutoGluon will save models to "c:\big20\git\big20-ML-project2-team3\MercariPriceSuggestion\src\autogluon_models"
Train Data Rows:    1482535
Train Data Columns: 6
Label Column:       price
AutoGluon infers your prediction problem is: 'regression' (because dtype of label-column == float and label-values can't be converted to int).


Train shape: (1482535, 7)
Test shape: (693359, 6)


	Label info (max, min, mean, stddev): (2009.0, 0.0, 26.73752, 38.58607)
	If 'regression' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify problem_type as one of: ['binary', 'multiclass', 'regression', 'quantile'])
Problem Type:       regression
Preprocessing data ...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    25358.47 MB
	Train Data (Original)  Memory Usage: 656.23 MB (2.6% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 1 features to boolean dtype as they only contain 2 unique values.
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fi

[1000]	valid_set's rmse: 27.3717
[2000]	valid_set's rmse: 27.0812
[3000]	valid_set's rmse: 26.8878
[4000]	valid_set's rmse: 26.79
[5000]	valid_set's rmse: 26.7255
[6000]	valid_set's rmse: 26.6453
[7000]	valid_set's rmse: 26.581
[8000]	valid_set's rmse: 26.548
[9000]	valid_set's rmse: 26.5283
[10000]	valid_set's rmse: 26.4915


	-26.4866	 = Validation score   (-root_mean_squared_error)
	265.97s	 = Training   runtime
	3.28s	 = Validation runtime
Fitting model: LightGBM ... Training model for up to 2773.50s of the 2773.50s of remaining time.
	Fitting with cpus=6, gpus=0, mem=8.5/21.1 GB


[1000]	valid_set's rmse: 27.0446
[2000]	valid_set's rmse: 26.8246
[3000]	valid_set's rmse: 26.7282
[4000]	valid_set's rmse: 26.6734
[5000]	valid_set's rmse: 26.6402
[6000]	valid_set's rmse: 26.6346


	-26.6195	 = Validation score   (-root_mean_squared_error)
	194.89s	 = Training   runtime
	1.67s	 = Validation runtime
Fitting model: RandomForestMSE ... Training model for up to 2576.30s of the 2576.30s of remaining time.
	Fitting with cpus=12, gpus=0, mem=0.9/21.0 GB
	-29.5057	 = Validation score   (-root_mean_squared_error)
	2130.99s	 = Training   runtime
	0.11s	 = Validation runtime
Fitting model: CatBoost ... Training model for up to 445.12s of the 445.12s of remaining time.
	Fitting with cpus=6, gpus=0, mem=9.3/20.9 GB
	Ran out of time, early stopping on iteration 354.
	-29.635	 = Validation score   (-root_mean_squared_error)
	443.95s	 = Training   runtime
	0.32s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ... Training model for up to 360.00s of the 0.68s of remaining time.
	Ensemble Weights: {'LightGBMXT': 0.5, 'LightGBM': 0.409, 'RandomForestMSE': 0.091}
	-26.2605	 = Validation score   (-root_mean_squared_error)
	0.02s	 = Training   runtime
	0.0s	 = Validation runt

                 model  score_val              eval_metric  pred_time_val  \
0  WeightedEnsemble_L2 -26.260456  root_mean_squared_error       5.052470   
1           LightGBMXT -26.486573  root_mean_squared_error       3.278735   
2             LightGBM -26.619528  root_mean_squared_error       1.666801   
3      RandomForestMSE -29.505658  root_mean_squared_error       0.105933   
4             CatBoost -29.634966  root_mean_squared_error       0.318119   

      fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  \
0  2591.858673                0.001000           0.020005            2   
1   265.965791                3.278735         265.965791            1   
2   194.886942                1.666801         194.886942            1   
3  2130.985935                0.105933        2130.985935            1   
4   443.949869                0.318119         443.949869            1   

   can_infer  fit_order  
0       True          5  
1       True          1  
2       True  